# Legacy CNN training


In [ ]:
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

from pathlib import Path


In [ ]:
run_dir = Path('/u/akazantsev/ml_dl_for_rfi_mitigation/runs/prepublic_run')

metadata_path = run_dir / 'B0531+21_59000_48386_subset_channels_meta.csv'
full_time_series_path = run_dir / 'B0531+21_59000_48386_channels.npy'
split_path = run_dir / 'split_indices.npz'
checkpoint_path = run_dir / 'best_model_by_acc.pt'

for input_path in (metadata_path, full_time_series_path, split_path):
    if not input_path.is_file():
        raise FileNotFoundError(f'Required input does not exist: {input_path}')

if checkpoint_path.exists():
    raise FileExistsError(f'Checkpoint already exists: {checkpoint_path}')

metadata = pd.read_csv(metadata_path, keep_default_na=False)

required_columns = {'sample_index', 'label'}
missing_columns = required_columns.difference(metadata.columns)
if missing_columns:
    raise ValueError(f'Metadata columns are missing: {sorted(missing_columns)}')

full_channel_time_series = np.load(full_time_series_path, mmap_mode='r')
if full_channel_time_series.ndim != 2 or full_channel_time_series.shape[1] != 256:
    raise ValueError('Full channel-time-series array must have shape (n_rows, 256)')

source_row_indices = metadata['sample_index'].to_numpy(dtype=np.int64)
if source_row_indices.min() < 0 or source_row_indices.max() >= len(full_channel_time_series):
    raise ValueError('sample_index is outside the full channel-time-series array')
if len(np.unique(source_row_indices)) != len(source_row_indices):
    raise ValueError('Subset metadata contains duplicate sample_index values')

label_names = metadata['label'].to_numpy()
if set(np.unique(label_names)) != {'NBRFI', 'None'}:
    raise ValueError('Subset metadata must contain only NBRFI and None labels')

with np.load(split_path) as split_data:
    expected_split_names = {'train_idx', 'val_idx', 'test_idx'}
    if set(split_data.files) != expected_split_names:
        raise ValueError(f'Split file must contain {sorted(expected_split_names)}')
    train_indices = split_data['train_idx'].astype(np.int64)
    validation_indices = split_data['val_idx'].astype(np.int64)
    test_indices = split_data['test_idx'].astype(np.int64)

all_split_indices = np.concatenate((train_indices, validation_indices, test_indices))
if all_split_indices.min() < 0 or all_split_indices.max() >= len(metadata):
    raise ValueError('Split indices are outside subset metadata')
if len(np.unique(all_split_indices)) != len(metadata):
    raise ValueError('Split indices must cover every subset row exactly once')

labels = (label_names == 'NBRFI').astype(np.float32)
class_rows = np.bincount(labels.astype(np.int64), minlength=2)
if not np.array_equal(class_rows, np.array([10_000, 10_000])):
    raise ValueError(f'Expected 10,000 rows per class, got {class_rows.tolist()}')

print('Full channel-time-series shape:', full_channel_time_series.shape)
print('Subset rows:', len(metadata))
print('Class rows [None, NBRFI]:', class_rows.tolist())
print('Split rows:', len(train_indices), len(validation_indices), len(test_indices))


In [ ]:
class FrequencyChannelDataset(Dataset):
    """Load one selected frequency-channel time series and its binary label."""

    def __init__(self, full_channel_time_series, labels, source_row_indices, subset_indices):
        self.full_channel_time_series = full_channel_time_series
        self.labels = labels
        self.source_row_indices = source_row_indices
        self.subset_indices = subset_indices

    def __len__(self):
        return len(self.subset_indices)

    def __getitem__(self, item_index):
        subset_index = self.subset_indices[item_index]
        source_row_index = self.source_row_indices[subset_index]

        frequency_channel_time_series = np.asarray(
            self.full_channel_time_series[source_row_index], dtype=np.float32
        )
        if frequency_channel_time_series.shape != (256,):
            raise ValueError('Each frequency-channel time series must contain 256 samples')

        # Preserve the reported per-channel maximum normalization.
        maximum = np.max(np.abs(frequency_channel_time_series))
        if maximum < 1e-8:
            maximum = 1.0
        frequency_channel_time_series = frequency_channel_time_series / maximum

        return (
            torch.from_numpy(frequency_channel_time_series).unsqueeze(0),
            torch.tensor(self.labels[subset_index], dtype=torch.float32),
        )


In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError('CUDA is required for the PUNCH CNN training run')

device = torch.device('cuda')
batch_size = 256

train_dataset = FrequencyChannelDataset(
    full_channel_time_series, labels, source_row_indices, train_indices
)
validation_dataset = FrequencyChannelDataset(
    full_channel_time_series, labels, source_row_indices, validation_indices
)
test_dataset = FrequencyChannelDataset(
    full_channel_time_series, labels, source_row_indices, test_indices
)

# Keep one memmap owner during portable training.
loader_options = {'batch_size': batch_size, 'num_workers': 0, 'pin_memory': True}
train_loader = DataLoader(train_dataset, shuffle=True, **loader_options)
validation_loader = DataLoader(validation_dataset, shuffle=False, **loader_options)
test_loader = DataLoader(test_dataset, shuffle=False, **loader_options)

first_batch, first_batch_labels = next(iter(train_loader))
if first_batch.shape != (batch_size, 1, 256) or not torch.isfinite(first_batch).all():
    raise ValueError('Training batch is invalid')

print('Device:', torch.cuda.get_device_name(device))
print('Training batch shape:', tuple(first_batch.shape))


In [ ]:
class CNN1D_RFI_256_logits(nn.Module):
    """Classify one normalized 256-sample frequency-channel time series."""

    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv1d(1, 64, kernel_size=7)
        self.pool1 = nn.MaxPool1d(kernel_size=2, stride=2)
        self.conv2 = nn.Conv1d(64, 128, kernel_size=7)
        self.pool2 = nn.MaxPool1d(kernel_size=2, stride=2)
        self.conv3 = nn.Conv1d(128, 256, kernel_size=10)
        self.pool3 = nn.MaxPool1d(kernel_size=2, stride=2)
        self.flatten = nn.Flatten()
        self.dropout = nn.Dropout(p=0.5)
        self.fc1 = nn.Linear(6400, 256)
        self.fc2 = nn.Linear(256, 1)

    def forward(self, inputs):
        outputs = self.pool1(F.relu(self.conv1(inputs)))
        outputs = F.relu(self.conv2(outputs))
        # Preserve the 6,400-input dense layer in the reported architecture.
        outputs = F.pad(outputs, (0, 1))
        outputs = self.pool2(outputs)
        outputs = self.pool3(F.relu(self.conv3(outputs)))
        outputs = self.flatten(outputs)
        outputs = self.dropout(F.relu(self.fc1(outputs)))
        return self.fc2(outputs)


model = CNN1D_RFI_256_logits().to(device)
parameter_count = sum(parameter.numel() for parameter in model.parameters())
if parameter_count != 2_024_833:
    raise ValueError(f'Unexpected trainable parameter count: {parameter_count}')
if model(torch.zeros(1, 1, 256, device=device)).shape != (1, 1):
    raise ValueError('CNN output shape is invalid')

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print('Trainable parameters:', parameter_count)


In [ ]:
def run_one_epoch(data_loader, training):
    """Run one training or evaluation epoch and return loss and accuracy."""
    model.train(training)

    total_loss = 0.0
    correct_rows = 0
    total_rows = 0

    for batch_inputs, batch_labels in data_loader:
        batch_inputs = batch_inputs.to(device, non_blocking=True)
        batch_labels = batch_labels.to(device, non_blocking=True).unsqueeze(1)

        if training:
            optimizer.zero_grad()

        with torch.set_grad_enabled(training):
            logits = model(batch_inputs)
            loss = criterion(logits, batch_labels)

            if training:
                loss.backward()
                optimizer.step()

        total_loss += loss.item() * len(batch_labels)
        predictions = (torch.sigmoid(logits) >= 0.5).to(torch.int64)
        correct_rows += (predictions == batch_labels.to(torch.int64)).sum().item()
        total_rows += len(batch_labels)

    return total_loss / total_rows, correct_rows / total_rows


In [ ]:
n_epochs = 50
best_validation_accuracy = -np.inf

epoch_progress = tqdm(range(1, n_epochs + 1), desc='Training', unit='epoch')

for epoch_number in epoch_progress:
    train_loss, train_accuracy = run_one_epoch(train_loader, training=True)
    validation_loss, validation_accuracy = run_one_epoch(validation_loader, training=False)

    # Select the checkpoint by validation accuracy, as in the historical run.
    if validation_accuracy > best_validation_accuracy:
        best_validation_accuracy = validation_accuracy
        torch.save(
            {
                'epoch': epoch_number,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'validation_loss': validation_loss,
                'validation_accuracy': validation_accuracy,
            },
            checkpoint_path,
        )

    epoch_progress.set_postfix(
        train_loss=f'{train_loss:.6f}',
        train_accuracy=f'{train_accuracy:.4f}',
        validation_loss=f'{validation_loss:.6f}',
        validation_accuracy=f'{validation_accuracy:.4f}',
    )


In [ ]:
checkpoint = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])

test_loss, test_accuracy = run_one_epoch(test_loader, training=False)

print('Best checkpoint epoch:', checkpoint['epoch'])
print('Best validation accuracy:', checkpoint['validation_accuracy'])
print('Test loss:', test_loss)
print('Test accuracy:', test_accuracy)
